In [21]:
# ── Imports ───────────────────────────────────────────────────────
import os
import functools
import pandas as pd
import numpy as np
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
import pyro.infer.autoguide as autoguide
from pyro.optim import Adam
from sklearn.preprocessing import RobustScaler
from scipy.stats import spearmanr
import polars as pl
from pybaseballstats.statcast import pitch_by_pitch_data
from pybaseballstats.utils.retrosheet_utils import _get_people_data

# ── Season Config ─────────────────────────────────────────────────
PRED_SEASON = 2026          # season to predict
TEST_SEASON = PRED_SEASON - 1   # held-out season for backtesting

# ── Feature Sets ──────────────────────────────────────────────────
DECAY_FEATURES  = ['K%', 'BB%', 'GB%', 'CSW%', 'FIP']
RECENT_FEATURES = ['IP_per_G', 'start_IP_per_GS', 'relief_IP_per_G', 'HLDSV_per_G', 'RS/9']
LAMBDA = 0.5 ** (1/3)

SP_FEATURES = (
    [f'{f}_decay'  for f in ['K%', 'BB%', 'GB%', 'CSW%', 'FIP']] +
    [f'{f}_recent' for f in ['IP_per_G', 'start_IP_per_GS', 'RS/9']] +
    ['age', 'age_sq', 'delta_IP_per_G']
)
RP_FEATURES = (
    [f'{f}_decay'  for f in ['K%', 'BB%', 'GB%', 'CSW%', 'FIP']] +
    [f'{f}_recent' for f in ['IP_per_G', 'relief_IP_per_G', 'HLDSV_per_G', 'RS/9']] +
    ['age', 'age_sq', 'delta_IP_per_G']
)

SEASON_GAMES = {yr: (60 if yr == 2020 else 162) for yr in range(2019, TEST_SEASON + 1)}
COUNTING_COLS = ['IP', 'SO', 'HR', 'H', 'ER', 'BB', 'HBP', 'WP', 'BK', 'W', 'L',
                 'BS', 'HLD', 'SV', 'Start-IP', 'Relief-IP', 'G', 'GS']

_STATCAST_CACHE_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'draft-tools', 'data', 'statcast_cache')

# ── Data Pipeline (pybaseballstats) ──────────────────────────────

def _fetch_season_raw(yr):
    os.makedirs(_STATCAST_CACHE_DIR, exist_ok=True)
    cache_path = os.path.join(_STATCAST_CACHE_DIR, f'{yr}.parquet')
    is_current = (yr == pd.Timestamp.now().year)
    if os.path.exists(cache_path) and not is_current:
        return pd.read_parquet(cache_path)

    # Download month-by-month so transient HTTP errors on individual chunks
    # don't abort the entire season.  Baseball Savant returns 502s under load,
    # and pybaseballstats refuses to return partial data for a single request,
    # so we split the season into monthly windows and skip any that fail.
    import datetime
    season_end = datetime.date(yr, 10, 31)
    month_starts = [datetime.date(yr, m, 1) for m in [3, 4, 5, 6, 7, 8, 9, 10]]
    chunks = []
    for i, start in enumerate(month_starts):
        end = (month_starts[i + 1] - datetime.timedelta(days=1)
               if i + 1 < len(month_starts) else season_end)
        if start > datetime.date.today():
            break
        end = min(end, datetime.date.today())
        max_retries, delay = 3, 10
        for attempt in range(1, max_retries + 1):
            try:
                result = pitch_by_pitch_data(
                    start_date=start.strftime('%Y-%m-%d'),
                    end_date=end.strftime('%Y-%m-%d'),
                    force_collect=True,
                    show_progress=False,
                )
                if result is not None:
                    chunks.append(result.to_pandas())
                break
            except RuntimeError as e:
                if attempt < max_retries:
                    import time
                    print(f'  {yr}-{start.month:02d} attempt {attempt} failed (502), retrying in {delay}s...')
                    time.sleep(delay)
                    delay *= 2
                else:
                    print(f'  Warning: {yr}-{start.month:02d} failed after {max_retries} attempts, skipping.')

    if not chunks:
        return pd.DataFrame()
    df = pd.concat(chunks, ignore_index=True)
    if not df.empty:
        # Monthly chunks can produce mixed-type object columns (e.g. float vs string
        # null representations) that PyArrow rejects on parquet write.  Coerce any
        # object column that is fully convertible to numeric, and cast the rest to str.
        for col in df.select_dtypes(include='object').columns:
            numeric = pd.to_numeric(df[col], errors='coerce')
            if numeric.notna().sum() >= df[col].notna().sum() * 0.5:
                df[col] = numeric
            else:
                df[col] = df[col].astype(str).where(df[col].notna(), other=None)
        df.to_parquet(cache_path, index=False)
    return df


@functools.lru_cache(maxsize=1)
def _pitcher_id_map():
    # Returns {mlbam: name} — MLBAM is the player key throughout
    people = _get_people_data()
    df = (people
          .filter(pl.col('key_mlbam').is_not_null())
          .select(['key_mlbam', 'name_first', 'name_last'])
          .to_pandas())
    df['Name'] = (df['name_first'].fillna('') + ' ' + df['name_last'].fillna('')).str.strip().str.title()
    return {int(row.key_mlbam): row.Name for row in df.itertuples(index=False)}


def _classify_events(df):
    df = df.copy()
    df['_SO']  = df['events'].isin(['strikeout', 'strikeout_double_play'])
    df['_BB']  = df['events'].isin(['walk', 'intent_walk'])
    df['_H']   = df['events'].isin(['single', 'double', 'triple', 'home_run'])
    df['_HBP'] = df['events'] == 'hit_by_pitch'
    df['_WP']  = df['events'] == 'wild_pitch'
    df['_BK']  = df['events'] == 'balk'
    df['_HR']  = df['events'] == 'home_run'
    df['_gb']  = df['bb_type'] == 'ground_ball'
    df['_bip'] = df['bb_type'].notna()
    df['_csw'] = df['description'].isin([
        'called_strike', 'swinging_strike', 'swinging_strike_blocked',
        'missed_bunt', 'swinging_pitchout',
    ])
    df['_PA']  = df['events'].notna()
    out_map = {
        'strikeout': 1,              'strikeout_double_play': 2,
        'field_out': 1,              'force_out': 1,
        'fielders_choice_out': 1,    'sac_fly': 1,
        'sac_bunt': 1,               'sac_bunt_double_play': 2,
        'sac_fly_double_play': 2,    'grounded_into_double_play': 2,
        'double_play': 2,            'triple_play': 3,
    }
    df['_outs'] = df['events'].map(out_map).fillna(0)
    return df


def _game_decisions(game_df):
    g = game_df.sort_values(['at_bat_number', 'pitch_number'])
    last_row = g.iloc[-1]
    home_won = last_row['post_home_score'] > last_row['post_away_score']

    stints = {}
    for pid, p_df in g.groupby('pitcher'):
        f, l = p_df.iloc[0], p_df.iloc[-1]
        team_home = (f['inning_topbot'] == 'Top')
        stints[pid] = dict(
            team_won     = (team_home == home_won),
            entry_inning = int(f['inning']),
            entry_lead   = f['fld_score'] - f['bat_score'],
            exit_lead    = l['post_fld_score'] - l['post_bat_score'],
            runners      = any(pd.notna(f[c]) for c in ['on_1b', 'on_2b', 'on_3b']),
            last_ab      = p_df['at_bat_number'].max(),
        )

    dec = {pid: dict(W=0, L=0, SV=0, HLD=0, BS=0) for pid in stints}

    winning_relievers = [
        pid for pid, s in stints.items()
        if s['team_won'] and s['entry_inning'] > 1
    ]
    closer = (max(winning_relievers, key=lambda p: stints[p]['last_ab'])
              if winning_relievers else None)

    for pid, s in stints.items():
        if s['entry_inning'] == 1:
            continue
        in_sv_sit = s['entry_lead'] > 0 and (s['entry_lead'] <= 3 or s['runners'])
        if not in_sv_sit:
            continue
        if pid == closer:
            dec[pid]['SV'] = 1
        elif s['team_won'] and s['exit_lead'] > 0:
            dec[pid]['HLD'] = 1
        elif s['exit_lead'] <= 0:
            dec[pid]['BS'] = 1

    win_given = False
    for pid, s in sorted(stints.items(), key=lambda x: x[1]['entry_inning']):
        if not s['team_won']:
            continue
        if s['entry_inning'] == 1 and s['exit_lead'] > 0:
            dec[pid]['W'] = 1; win_given = True; break
    if not win_given:
        for pid, s in sorted(stints.items(), key=lambda x: x[1]['entry_inning']):
            if not s['team_won'] or s['entry_inning'] == 1 or pid == closer:
                continue
            if s['exit_lead'] > 0:
                dec[pid]['W'] = 1; win_given = True; break
    if not win_given and closer and stints[closer]['team_won']:
        dec[closer]['W'] = 1

    for pid, s in sorted(stints.items(), key=lambda x: x[1]['entry_inning']):
        if s['team_won']:
            continue
        if s['exit_lead'] < 0:
            dec[pid]['L'] = 1; break

    return dec


def pull_seasons(seasons):
    id_map = _pitcher_id_map()
    season_dfs = []

    for yr in seasons:
        raw = _fetch_season_raw(yr)
        if raw.empty:
            continue
        raw['game_date'] = pd.to_datetime(raw['game_date'], errors='coerce')
        raw = _classify_events(raw)
        # Filter to regular season only — spring training ('S'), exhibitions ('E'),
        # and postseason ('D','L','F','W') are excluded since they don't count for fantasy.
        if 'game_type' in raw.columns:
            raw = raw[raw['game_type'] == 'R']

        # Sort chronologically so .first()/.last() aggregates reflect entry/exit
        # not desc-sorted Baseball Savant order (which would make entry_inning wrong).
        raw = raw.sort_values(['game_pk', 'at_bat_number', 'pitch_number'])

        gp = (raw.groupby(['game_pk', 'pitcher'])
              .agg(
                  SO           = ('_SO',          'sum'),
                  BB           = ('_BB',          'sum'),
                  HR           = ('_HR',          'sum'),
                  H            = ('_H',           'sum'),
                  HBP          = ('_HBP',         'sum'),
                  WP           = ('_WP',          'sum'),
                  BK           = ('_BK',          'sum'),
                  gb           = ('_gb',          'sum'),
                  bip          = ('_bip',         'sum'),
                  csw          = ('_csw',         'sum'),
                  n_pitches    = ('description',  'count'),
                  PA           = ('_PA',          'sum'),
                  outs         = ('_outs',        'sum'),
                  entry_inning = ('inning',        'first'),
                  age          = ('age_pit',       'first'),
                  fld_entry    = ('fld_score',     'first'),
                  bat_entry    = ('bat_score',     'first'),
                  fld_exit     = ('post_fld_score','last'),
                  bat_exit     = ('post_bat_score','last'),
                  name         = ('player_name',  'first'),
              )
              .reset_index())

        gp['IP']       = gp['outs'] / 3
        gp['R']        = (gp['bat_exit'] - gp['bat_entry']).clip(lower=0)
        gp['RS']       = (gp['fld_exit'] - gp['fld_entry']).clip(lower=0)
        gp['is_start'] = gp['entry_inning'] == 1

        dec_rows = []
        for game_pk, game_df in raw.groupby('game_pk'):
            for pid, d in _game_decisions(game_df).items():
                dec_rows.append({'game_pk': game_pk, 'pitcher': pid, **d})
        dec_df = pd.DataFrame(dec_rows)
        gp = gp.merge(dec_df, on=['game_pk', 'pitcher'], how='left')
        for col in ['W', 'L', 'SV', 'HLD', 'BS']:
            gp[col] = gp[col].fillna(0).astype(int)

        start_ip  = (gp[gp['is_start']]
                     .groupby('pitcher')['IP'].sum()
                     .rename('Start-IP'))
        relief_ip = (gp[~gp['is_start']]
                     .groupby('pitcher')['IP'].sum()
                     .rename('Relief-IP'))

        season = (gp.groupby('pitcher')
                  .agg(
                      SO        = ('SO',        'sum'),
                      BB        = ('BB',        'sum'),
                      HR        = ('HR',        'sum'),
                      H         = ('H',         'sum'),
                      ER        = ('R',         'sum'),
                      HBP       = ('HBP',       'sum'),
                      WP        = ('WP',        'sum'),
                      BK        = ('BK',        'sum'),
                      gb        = ('gb',        'sum'),
                      bip       = ('bip',       'sum'),
                      csw       = ('csw',       'sum'),
                      n_pitches = ('n_pitches', 'sum'),
                      PA        = ('PA',        'sum'),
                      IP        = ('IP',        'sum'),
                      G         = ('game_pk',   'nunique'),
                      GS        = ('is_start',  'sum'),
                      W         = ('W',         'sum'),
                      L         = ('L',         'sum'),
                      SV        = ('SV',        'sum'),
                      HLD       = ('HLD',       'sum'),
                      BS        = ('BS',        'sum'),
                      RS        = ('RS',        'sum'),
                      Age       = ('age',       'mean'),
                      Name      = ('name',      'first'),
                  )
                  .reset_index()
                  .join(start_ip, on='pitcher')
                  .join(relief_ip, on='pitcher'))

        season['Start-IP']  = season['Start-IP'].fillna(0)
        season['Relief-IP'] = season['Relief-IP'].fillna(0)
        season['Age']       = season['Age'].round().astype('Int64')

        season['K%']   = season['SO']  / season['PA'].clip(lower=1)
        season['BB%']  = season['BB']  / season['PA'].clip(lower=1)
        season['GB%']  = season['gb']  / season['bip'].clip(lower=1)
        season['CSW%'] = season['csw'] / season['n_pitches'].clip(lower=1)
        # FIP: scale-invariant ERA proxy that captures HR, BB, HBP, K rates
        FIP_C = 3.10  # standard FIP constant
        season['FIP'] = ((13*season['HR'] + 3*(season['BB']+season['HBP']) - 2*season['SO'])
                         / season['IP'].clip(lower=1) + FIP_C)
        season['RS/9'] = season['RS']  / season['IP'].clip(lower=1) * 9

        # statcast player_name is "Last, First" — convert to "First Last" before id_map override
        def _fix_statcast_name(n):
            if pd.isna(n) or ',' not in str(n):
                return n
            last, first = str(n).split(',', 1)
            return f'{first.strip()} {last.strip()}'.title()
        season['Name'] = season['Name'].apply(_fix_statcast_name)
        season['Name'] = season.apply(
            lambda row: id_map.get(int(row['pitcher'])) or row['Name'],
            axis=1,
        )
        season['Season'] = yr
        season_dfs.append(season)

    if not season_dfs:
        return pd.DataFrame()

    result = pd.concat(season_dfs, ignore_index=True)
    return result[result['IP'] >= 1].reset_index(drop=True)


def normalize_season_length(df):
    df = df.copy()
    df['season_factor'] = df['Season'].map(SEASON_GAMES) / 162
    for col in COUNTING_COLS:
        df[col] = df[col] / df['season_factor']
    return df

def add_role_proxies(df):
    df['IP_per_G']        = df['IP'] / df['G']
    df['start_IP_per_GS'] = np.where(df['GS'] == 0, 0, df['Start-IP'] / df['GS'])
    df['relief_IP_per_G'] = df['Relief-IP'] / df['G']
    df['HLDSV_per_G']     = (df['HLD'] + df['SV']) / df['G']
    df['is_SP']           = (df['GS'] >= 10) & (df['start_IP_per_GS'] >= 5)
    return df

def add_fantasy_points(df):
    base = (df['IP']*2.49 + df['SO']*0.75 + df['H']*-0.75 + df['ER']*-2.75 +
            df['BB']*-0.5 + df['HBP']*-0.75 + df['WP']*-0.75 + df['BK']*-1.5 +
            df['W']*2.0 + df['L']*-1.0 + df['BS']*-2.0)
    rp_bonus = df['HLD']*2.25 + df['SV']*2.25
    df['fantasy_pts']        = np.where(df['is_SP'], base, base + rp_bonus)
    df['fantasy_pts_per_IP'] = df['fantasy_pts'] / np.clip(df['IP'], 10, None)
    return df

def build_rolling_windows(df):
    rows = []
    df = add_role_proxies(df)
    df = add_fantasy_points(df)
    for pid in df['pitcher'].unique():
        pdf = df[df['pitcher'] == pid].sort_values('Season')
        for target_season in pdf['Season'].values:
            past = pdf[pdf['Season'] < target_season].tail(3)
            if len(past) == 0:
                continue
            w = np.array([LAMBDA**(len(past)-1-j) for j in range(len(past))])
            w /= w.sum()
            row = {}
            for feat in DECAY_FEATURES:
                vals = past[feat].values
                ww = w.copy(); ww[np.isnan(vals)] = 0
                row[f'{feat}_decay'] = np.nan if ww.sum()==0 else (vals * (ww/ww.sum())).sum()
            for feat in RECENT_FEATURES:
                row[f'{feat}_recent'] = past[feat].iloc[-1]
            row['age']          = past['Age'].iloc[-1] + 1
            row['age_sq']       = row['age'] ** 2
            row['experience']   = len(past)
            row['is_SP']        = past.iloc[-1]['is_SP']
            row['delta_IP_per_G'] = np.diff(past['IP_per_G'].tail(2).values)[0] if len(past)>1 else 0
            tgt = pdf[pdf['Season']==target_season].iloc[0]
            sf  = SEASON_GAMES.get(target_season, 162) / 162
            row['target_fantasy_pts_per_IP'] = tgt['fantasy_pts_per_IP']
            row['target_IP']  = tgt['IP'] * sf
            row['pitcher']    = pid
            row['season']     = target_season
            rows.append(row)
    return pd.DataFrame(rows)

def build_inference_rows(df, role='SP'):
    recent = (['IP_per_G', 'start_IP_per_GS', 'RS/9'] if role=='SP'
              else ['IP_per_G', 'relief_IP_per_G', 'HLDSV_per_G', 'RS/9'])
    rows = []
    df = add_role_proxies(df)
    df = add_fantasy_points(df)
    for pid in df['pitcher'].unique():
        pdf = df[df['pitcher'] == pid].sort_values('Season')
        mr  = pdf.iloc[-1]
        if role=='SP' and not mr['is_SP']: continue
        if role=='RP' and mr['is_SP']:     continue
        past = pdf.tail(3)
        if len(past) == 0: continue
        w = np.array([LAMBDA**(len(past)-1-j) for j in range(len(past))])
        w /= w.sum()
        row = {}
        for feat in DECAY_FEATURES:
            vals = past[feat].values
            ww = w.copy(); ww[np.isnan(vals)] = 0
            row[f'{feat}_decay'] = np.nan if ww.sum()==0 else (vals*(ww/ww.sum())).sum()
        for feat in recent:
            row[f'{feat}_recent'] = past[feat].iloc[-1]
        row['name']          = mr['Name']
        row['age']           = past['Age'].iloc[-1] + 1
        row['age_sq']        = row['age'] ** 2
        row['experience']    = len(past)
        row['is_SP']         = mr['is_SP']
        row['delta_IP_per_G'] = np.diff(past['IP_per_G'].tail(2).values)[0] if len(past)>1 else 0
        row['pitcher']       = pid
        rows.append(row)
    return pd.DataFrame(rows)

# ── Splits & Tensors ─────────────────────────────────────────────
def train_test_split(windows_df, test_season=None):
    if test_season is None:
        return windows_df, None
    return (windows_df[windows_df['season'] < test_season],
            windows_df[windows_df['season'] == test_season])

def split_by_role(windows_df):
    return (windows_df[windows_df['is_SP']],
            windows_df[~windows_df['is_SP']])

def prepare_tensors(role_train, role_df, features, scaler=None):
    encoder  = {pid: idx for idx, pid in enumerate(role_train['pitcher'].unique())}
    role_df  = (role_df
                .dropna(subset=features)
                [lambda d: d['pitcher'].isin(encoder)]
                .reset_index(drop=True))
    if scaler is None:
        scaler = RobustScaler()
        X = torch.tensor(scaler.fit_transform(role_df[features]), dtype=torch.float32)
    else:
        X = torch.tensor(scaler.transform(role_df[features]), dtype=torch.float32)
    pitcher_idx = torch.tensor(role_df['pitcher'].map(encoder).values, dtype=torch.long)
    y_pts = torch.tensor(role_df['target_fantasy_pts_per_IP'].values, dtype=torch.float32)
    y_IP  = torch.tensor(role_df['target_IP'].values, dtype=torch.float32)
    return X, pitcher_idx, y_pts, y_IP, scaler, encoder, role_df

# ── Model ────────────────────────────────────────────────────────
def model(X, pitcher_idx, n_pitchers, y_pts_per_IP=None, y_IP=None, role='SP'):
    n_features     = X.shape[1]
    ip_prior_mean  = 5.0 if role=='SP' else 3.7
    ip_sigma_prior = 30.0 if role=='SP' else 20.0

    alpha_pts   = pyro.sample('alpha_pts',  dist.Normal(0, 1))
    alpha_IP    = pyro.sample('alpha_IP',   dist.Normal(ip_prior_mean, 1))
    beta_pts    = pyro.sample('beta_pts',   dist.Normal(0,1).expand([n_features]).to_event(1))
    beta_IP     = pyro.sample('beta_IP',    dist.Normal(0,1).expand([n_features]).to_event(1))
    sigma_u_pts = pyro.sample('sigma_u_pts', dist.HalfNormal(0.5))
    sigma_u_IP  = pyro.sample('sigma_u_IP',  dist.HalfNormal(1))

    # sigma_pts is global — per-pitcher quality noise would inflate uncertainty for
    # pitchers with variable career histories (e.g. early-career struggles before
    # breaking out). sigma_IP stays per-pitcher since IP workload is genuinely
    # individual (role changes, injury history, team usage patterns).
    sigma_pts = pyro.sample('sigma_pts', dist.HalfNormal(1))

    with pyro.plate('pitchers', n_pitchers):
        u_pts     = pyro.sample('u_pts',     dist.Normal(0, sigma_u_pts))
        u_IP      = pyro.sample('u_IP',      dist.Normal(0, sigma_u_IP))
        sigma_IP  = pyro.sample('sigma_IP',  dist.HalfNormal(ip_sigma_prior))

    with pyro.plate('obs', X.shape[0]):
        mu_pts = alpha_pts + u_pts[pitcher_idx] + (X * beta_pts).sum(-1)
        mu_IP  = torch.exp(
            (alpha_IP + u_IP[pitcher_idx] + (X * beta_IP).sum(-1)).clamp(-6, 6)
        ).clamp(min=1e-4)
        sig_IP = sigma_IP[pitcher_idx].clamp(min=1e-4)
        pyro.sample('y_pts', dist.Normal(mu_pts, sigma_pts),    obs=y_pts_per_IP)
        pyro.sample('y_IP',  dist.Gamma(mu_IP**2/sig_IP**2, mu_IP/sig_IP**2), obs=y_IP)

def sp_model(X, pitcher_idx, n_pitchers, y_pts_per_IP=None, y_IP=None):
    return model(X, pitcher_idx, n_pitchers, y_pts_per_IP, y_IP, role='SP')

def rp_model(X, pitcher_idx, n_pitchers, y_pts_per_IP=None, y_IP=None):
    return model(X, pitcher_idx, n_pitchers, y_pts_per_IP, y_IP, role='RP')

# ── Inference ────────────────────────────────────────────────────
def run_inference(model_fn, X, pitcher_idx, n_pitchers,
                  y_pts, y_IP, n_steps=3000, lr=0.01):
    torch.manual_seed(42)
    pyro.set_rng_seed(42)
    pyro.clear_param_store()
    guide = autoguide.AutoDiagonalNormal(model_fn)
    svi   = SVI(model_fn, guide, Adam({'lr': lr}), loss=Trace_ELBO())
    losses = []
    for step in range(n_steps):
        loss = svi.step(X, pitcher_idx, n_pitchers, y_pts, y_IP)
        losses.append(loss)
        if step % 500 == 0:
            print(f'  step {step:4d}  loss {loss:.1f}')
    return guide, losses

# ── Evaluation ───────────────────────────────────────────────────
def evaluate(guide, model_fn, X_test, pitcher_idx_test, n_pitchers, test_df):
    predictive = Predictive(model_fn, guide=guide, num_samples=1000)
    samples    = predictive(X_test, pitcher_idx_test, n_pitchers)
    total      = (samples['y_pts'] * samples['y_IP']).detach().numpy()
    mean_pts   = total.mean(axis=0)
    std_pts    = total.std(axis=0)
    p10        = np.percentile(total, 10, axis=0)
    p90        = np.percentile(total, 90, axis=0)
    score      = mean_pts / np.clip(std_pts, 10, None)
    actual     = (test_df['target_fantasy_pts_per_IP'] * test_df['target_IP']).values
    rankings = pd.DataFrame({
        'pitcher': test_df['pitcher'].values,
        'mean_pts': mean_pts, 'std_pts': std_pts,
        'score': score, 'p10': p10, 'p90': p90, 'actual_pts': actual
    }).sort_values('score', ascending=False)
    metrics = pd.DataFrame({
        'spearman_mean':  [spearmanr(mean_pts, actual).correlation],
        'spearman_score': [spearmanr(score,    actual).correlation],
        'coverage_80pct': [((actual >= p10) & (actual <= p90)).mean()]
    })
    return rankings, metrics, samples

# ── New-Season Prediction ─────────────────────────────────────────
def risk_adjusted_score(mean_pts, std_pts, alpha=0.5, floor=10):
    return mean_pts / np.clip(std_pts, floor, None) ** alpha

def predict_new_season(guide, model_fn, inference_df, features,
                       scaler, encoder, n_pitchers, alpha=0.5, floor=10):
    df = inference_df.copy()
    df['is_known'] = df['pitcher'].map(lambda p: p in encoder)
    df[features]   = df[features].fillna(0)
    df['pitcher_idx'] = df['pitcher'].map(encoder).fillna(n_pitchers).astype(int)

    X           = torch.tensor(scaler.transform(df[features]), dtype=torch.float32)
    pitcher_idx = torch.tensor(df['pitcher_idx'].values, dtype=torch.long)

    predictive = Predictive(model_fn, guide=guide, num_samples=1000)
    samples    = predictive(X, pitcher_idx, n_pitchers + 1)
    total      = (samples['y_pts'] * samples['y_IP']).detach().numpy()

    mean_pts = total.mean(axis=0)
    std_pts  = total.std(axis=0)
    p10      = np.percentile(total, 10, axis=0)
    p90      = np.percentile(total, 90, axis=0)
    score    = risk_adjusted_score(mean_pts, std_pts, alpha=alpha, floor=floor)

    rankings = pd.DataFrame({
        'name':     df['name'].values,
        'age':      df['age'].values,
        'mean_pts': mean_pts.round(1),
        'std_pts':  std_pts.round(1),
        'p10':      p10.round(1),
        'p90':      p90.round(1),
        'score':    score.round(3),
        'is_known': df['is_known'].values,
    }).sort_values('score', ascending=False).reset_index(drop=True)
    # Unknown pitchers all share the same prior slot — their predictions are
    # indistinguishable noise. Filter them from the final rankings.
    return rankings[rankings['is_known']].reset_index(drop=True)


In [22]:
# ── Data Construction Diagnostics ───────────────────────────────
# Run this cell to verify statistics are correctly built from statcast
# before running the full model pipeline.

import warnings
warnings.filterwarnings('ignore')

print("Loading 2024 season data...")
df_2024 = pull_seasons([2024])

if df_2024.empty:
    print("ERROR: No data returned for 2024")
else:
    print(f"  {len(df_2024)} pitcher-seasons loaded\n")

    # ── 1. Check Paul Skenes ──────────────────────────────────────
    skenes = df_2024[df_2024['Name'].str.contains('Skenes', case=False, na=False)]
    print("=== Paul Skenes 2024 ===")
    if skenes.empty:
        print("  NOT FOUND — searching via id_map...")
        id_map = _pitcher_id_map()
        skenes_id = next((k for k, v in id_map.items() if 'Skenes' in v), None)
        if skenes_id:
            print(f"  Found in id_map: {skenes_id} -> {id_map[skenes_id]}")
            skenes = df_2024[df_2024['pitcher'] == skenes_id]
            if skenes.empty:
                print(f"  MLBAM {skenes_id} in id_map but NOT in 2024 data")
            else:
                print(skenes[['pitcher','Name','IP','SO','BB','W','L','G','GS','K%','BB%','GB%','CSW%']].to_string(index=False))
        else:
            print("  Not in Chadwick id_map — likely a data gap; add to MANUAL_PITCHER_NAMES below")
    else:
        print(skenes[['pitcher','Name','IP','SO','BB','W','L','G','GS','K%','BB%','GB%','CSW%']].to_string(index=False))

    # ── 2. Elite 2024 SPs presence check ─────────────────────────
    elite_names = ['Tarik Skubal', 'Zack Wheeler', 'Chris Sale', 'Dylan Cease',
                   'Gerrit Cole', 'Logan Webb', 'Corbin Burnes', 'Blake Snell']
    print("\n=== Elite SP presence in 2024 data ===")
    for name in elite_names:
        row = df_2024[df_2024['Name'].str.contains(name.split()[-1], case=False, na=False)]
        if row.empty:
            print(f"  MISSING: {name}")
        else:
            r = row.iloc[0]
            print(f"  {r['Name']:25s}  IP={r['IP']:5.1f}  SO={r['SO']:3.0f}  K%={r['K%']:.3f}  W={r['W']:2.0f}  L={r['L']:2.0f}  GS={r['GS']:2.0f}")

    # ── 3. Top SPs by K% ─────────────────────────────────────────
    print("\n=== Top 15 qualified SPs by K% (IP >= 100) ===")
    sp_check = df_2024[df_2024['IP'] >= 100].copy()
    sp_check['ERA'] = (sp_check['ER'] / sp_check['IP'].clip(lower=1) * 9).round(2)
    sp_check = sp_check.sort_values('K%', ascending=False)
    print(sp_check[['Name','IP','SO','K%','BB%','GB%','W','L','ERA']].head(15).to_string(index=False))

    # ── 4. W/L/SV/HLD totals plausibility ────────────────────────
    print("\n=== W/L/SV/HLD totals across all 2024 pitchers ===")
    print(f"  Total W={df_2024['W'].sum():.0f}  L={df_2024['L'].sum():.0f}  "
          f"SV={df_2024['SV'].sum():.0f}  HLD={df_2024['HLD'].sum():.0f}  BS={df_2024['BS'].sum():.0f}")
    print("  (MLB 2024 had 2430 games -> ~2430 W and ~2430 L expected)")

    # ── 5. Inference rows: verify Skenes in SP pool ───────────────
    print("\n=== Building multi-year data + inference rows ===")
    df_all = pull_seasons(range(2019, 2026))
    df_all = normalize_season_length(df_all)
    inf_sp = build_inference_rows(df_all, role='SP')
    if inf_sp.empty:
        print("  No SP inference rows!")
    else:
        print(f"  {len(inf_sp)} SP inference rows total")
        print("\n  Top 20 SPs by recent IP/G:")
        print(inf_sp.sort_values('IP_per_G_recent', ascending=False)
              [['name','age','IP_per_G_recent','K%_decay','BB%_decay','GB%_decay']]
              .head(20).to_string(index=False))
        skenes_inf = inf_sp[inf_sp['name'].str.contains('Skenes', case=False, na=False)]
        if skenes_inf.empty:
            print("\n  WARNING: Paul Skenes not in SP inference rows")
        else:
            print(f"\n  Paul Skenes in inference rows:")
            print(skenes_inf[['name','age','IP_per_G_recent','K%_decay','BB%_decay']].to_string(index=False))


Loading 2024 season data...
  840 pitcher-seasons loaded

=== Paul Skenes 2024 ===
 pitcher        Name         IP  SO  BB  W  L  G  GS       K%      BB%      GB%     CSW%
694973.0 Paul Skenes 131.333333 170  32 13  3 23  23 0.330097 0.062136 0.522876 0.294311

=== Elite SP presence in 2024 data ===
  Tarik Skubal               IP=190.7  SO=228  K%=0.302  W=19  L= 7  GS=31
  Zack Wheeler               IP=198.7  SO=224  K%=0.284  W=16  L=10  GS=32
  Chris Sale                 IP=175.0  SO=225  K%=0.320  W=18  L= 5  GS=29
  Dylan Cease                IP=187.3  SO=224  K%=0.294  W=15  L=11  GS=33
  Gerrit Cole                IP= 94.0  SO= 99  K%=0.254  W= 7  L= 5  GS=17
  Jacob Webb                 IP= 56.0  SO= 58  K%=0.245  W= 2  L= 1  GS= 0
  Corbin Burnes              IP=191.7  SO=181  K%=0.230  W=18  L= 9  GS=32
  Blake Snell                IP=103.0  SO=145  K%=0.347  W= 8  L= 4  GS=20

=== Top 15 qualified SPs by K% (IP >= 100) ===
           Name         IP  SO       K%      BB%   

In [23]:
df = pull_seasons(range(2019, PRED_SEASON))
df = normalize_season_length(df)
windows_df = build_rolling_windows(df)
windows_df['relief_IP_per_G_recent'] = windows_df['relief_IP_per_G_recent'].fillna(0)

train, test = train_test_split(windows_df, test_season=TEST_SEASON)
sp_train, sp_test = split_by_role(train)[0], split_by_role(test)[0]
rp_train, rp_test = split_by_role(train)[1], split_by_role(test)[1]

print(f"Seasons loaded: {sorted(windows_df['season'].unique())}")
print(f"Backtest split: train < {TEST_SEASON}, test = {TEST_SEASON}")
print(f"SP — train {len(sp_train)}, test {len(sp_test)}")
print(f"RP — train {len(rp_train)}, test {len(rp_test)}")

Seasons loaded: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Backtest split: train < 2025, test = 2025
SP — train 408, test 126
RP — train 2340, test 566


# Starting Pitchers

In [24]:
# ── Backtest: train < TEST_SEASON, evaluate on TEST_SEASON ───────
X_sp_train, pitcher_idx_sp_train, y_pts_sp_train, y_IP_sp_train, \
    scaler_sp, sp_encoder, sp_train_df = \
    prepare_tensors(sp_train, sp_train, SP_FEATURES)

X_sp_test, pitcher_idx_sp_test, y_pts_sp_test, y_IP_sp_test, \
    _, _, sp_test_df = \
    prepare_tensors(sp_train, sp_test, SP_FEATURES, scaler=scaler_sp)

n_pitchers_sp = len(sp_encoder)
print(f"SP: {len(X_sp_train)} train obs, {len(X_sp_test)} test obs, {n_pitchers_sp} pitchers")

sp_guide, sp_losses = run_inference(
    sp_model, X_sp_train, pitcher_idx_sp_train, n_pitchers_sp,
    y_pts_sp_train, y_IP_sp_train)

sp_rankings, sp_metrics, sp_samples = evaluate(
    sp_guide, sp_model, X_sp_test, pitcher_idx_sp_test, n_pitchers_sp, sp_test_df)

print(sp_metrics.to_string(index=False))

SP: 408 train obs, 91 test obs, 220 pitchers
  step    0  loss 155368.5
  step  500  loss 3616.4
  step 1000  loss 3329.0
  step 1500  loss 3260.0
  step 2000  loss 3143.2
  step 2500  loss 3189.1
 spearman_mean  spearman_score  coverage_80pct
        0.3767        0.344609        0.736264


In [25]:
# ── SP Diagnostics ────────────────────────────────────────────────
total    = (sp_samples['y_pts'] * sp_samples['y_IP']).detach().numpy()
actual   = sp_test_df['target_fantasy_pts_per_IP'].values * sp_test_df['target_IP'].values

print(f"Predicted  mean={total.mean():.1f}  std={total.std(axis=0).mean():.1f}")
print(f"Actual     mean={actual.mean():.1f}  std={actual.std():.1f}")
print(f"80% interval width: {(np.percentile(total,90,axis=0) - np.percentile(total,10,axis=0)).mean():.1f}")

# baseline
print(f"\nBaseline Spearman (K%_decay):  {spearmanr(-sp_test_df['K%_decay'], actual).correlation:.3f}")
print(f"Model Spearman (mean_pts):     {spearmanr(total.mean(axis=0), actual).correlation:.3f}")

# feature importance
locs = pyro.get_param_store()['AutoDiagonalNormal.loc'].detach().numpy()
n_f  = len(SP_FEATURES)
print(pd.DataFrame({
    'feature': SP_FEATURES,
    'beta_pts': locs[2:2+n_f].round(3),
    'beta_IP':  locs[2+n_f:2+2*n_f].round(3)
}).sort_values('beta_pts', key=abs, ascending=False).to_string(index=False))

Predicted  mean=89.7  std=115.2
Actual     mean=141.1  std=103.0
80% interval width: 256.0

Baseline Spearman (K%_decay):  -0.343
Model Spearman (mean_pts):     0.377
               feature  beta_pts  beta_IP
              K%_decay     0.324    0.138
             BB%_decay    -0.128   -0.201
             FIP_decay    -0.105   -0.138
                   age    -0.088    0.170
        delta_IP_per_G     0.060    0.092
           RS/9_recent     0.052    0.037
start_IP_per_GS_recent    -0.044   -0.318
            CSW%_decay     0.031   -0.186
                age_sq     0.026   -0.084
       IP_per_G_recent    -0.017    0.098
             GB%_decay    -0.002    0.006


# SP: New Season Predictions

In [26]:
# ── Train on ALL seasons, predict PRED_SEASON ─────────────────────
print(f"Training SP on all data through {PRED_SEASON - 1}...")
train_all, _ = train_test_split(windows_df, test_season=None)
sp_train_all  = split_by_role(train_all)[0]

X_sp_all, pitcher_idx_sp_all, y_pts_sp_all, y_IP_sp_all, \
    scaler_sp_final, sp_encoder_final, _ = \
    prepare_tensors(sp_train_all, sp_train_all, SP_FEATURES)

n_pitchers_sp_all = len(sp_encoder_final)
# train with n_pitchers + 1: last slot is unobserved (prior = N(0, sigma_u))
sp_guide_final, sp_losses_final = run_inference(
    sp_model, X_sp_all, pitcher_idx_sp_all, n_pitchers_sp_all + 1,
    y_pts_sp_all, y_IP_sp_all)

sp_inference       = build_inference_rows(df, role='SP')
sp_rankings_new    = predict_new_season(
    sp_guide_final, sp_model, sp_inference,
    SP_FEATURES, scaler_sp_final, sp_encoder_final, n_pitchers_sp_all, alpha = 0.1)

print(f"\n=== Top 20 SP for {PRED_SEASON} ===")
print(sp_rankings_new.head(20).to_string(index=False))

Training SP on all data through 2025...
  step    0  loss 192770.1
  step  500  loss 4799.7
  step 1000  loss 4299.3
  step 1500  loss 4104.2
  step 2000  loss 4010.2
  step 2500  loss 3970.7

=== Top 20 SP for 2026 ===
              name  age   mean_pts    std_pts        p10        p90      score  is_known
     Logan Gilbert   29 288.100006 184.399994  71.300003 525.200012 170.972000      True
        Logan Webb   30 276.299988 221.500000  10.500000 559.599976 160.983994      True
       Paul Skenes   24 251.199997 163.399994  60.299999 464.700012 150.882996      True
      Tarik Skubal   30 250.000000 166.199997  81.000000 462.700012 149.944000      True
Cristopher Sánchez   30 219.399994 172.199997  22.700001 443.899994 131.082001      True
       Dylan Cease   31 216.100006 176.100006   9.600000 437.600006 128.820999      True
          Joe Ryan   30 197.500000 147.600006  32.000000 381.700012 119.870003      True
    Freddy Peralta   30 184.800003 157.100006   3.600000 393.899994 

# Relief Pitchers

In [27]:
# ── Backtest: train < TEST_SEASON, evaluate on TEST_SEASON ───────
X_rp_train, pitcher_idx_rp_train, y_pts_rp_train, y_IP_rp_train, \
    scaler_rp, rp_encoder, rp_train_df = \
    prepare_tensors(rp_train, rp_train, RP_FEATURES)

X_rp_test, pitcher_idx_rp_test, y_pts_rp_test, y_IP_rp_test, \
    _, _, rp_test_df = \
    prepare_tensors(rp_train, rp_test, RP_FEATURES, scaler=scaler_rp)

n_pitchers_rp = len(rp_encoder)
print(f"RP: {len(X_rp_train)} train obs, {len(X_rp_test)} test obs, {n_pitchers_rp} pitchers")

rp_guide, rp_losses = run_inference(
    rp_model, X_rp_train, pitcher_idx_rp_train, n_pitchers_rp,
    y_pts_rp_train, y_IP_rp_train)

rp_rankings, rp_metrics, rp_samples = evaluate(
    rp_guide, rp_model, X_rp_test, pitcher_idx_rp_test, n_pitchers_rp, rp_test_df)

print(rp_metrics.to_string(index=False))

RP: 2340 train obs, 414 test obs, 1053 pitchers
  step    0  loss 1300089.6
  step  500  loss 18231.5
  step 1000  loss 17407.4
  step 1500  loss 17016.6
  step 2000  loss 16495.8
  step 2500  loss 16196.2
 spearman_mean  spearman_score  coverage_80pct
      0.376509        0.354551        0.702899


In [28]:
# ── RP Diagnostics ────────────────────────────────────────────────
total  = (rp_samples['y_pts'] * rp_samples['y_IP']).detach().numpy()
actual = rp_test_df['target_fantasy_pts_per_IP'].values * rp_test_df['target_IP'].values

print(f"Predicted  mean={total.mean():.1f}  std={total.std(axis=0).mean():.1f}")
print(f"Actual     mean={actual.mean():.1f}  std={actual.std():.1f}")
print(f"80% interval width: {(np.percentile(total,90,axis=0) - np.percentile(total,10,axis=0)).mean():.1f}")

print(f"\nBaseline Spearman (K%_decay):  {spearmanr(-rp_test_df['K%_decay'], actual).correlation:.3f}")
print(f"Model Spearman (mean_pts):     {spearmanr(total.mean(axis=0), actual).correlation:.3f}")

locs = pyro.get_param_store()['AutoDiagonalNormal.loc'].detach().numpy()
n_f  = len(RP_FEATURES)
print(pd.DataFrame({
    'feature': RP_FEATURES,
    'beta_pts': locs[2:2+n_f].round(3),
    'beta_IP':  locs[2+n_f:2+2*n_f].round(3)
}).sort_values('beta_pts', key=abs, ascending=False).to_string(index=False))

Predicted  mean=29.4  std=57.3
Actual     mean=55.3  std=69.4
80% interval width: 108.7

Baseline Spearman (K%_decay):  -0.354
Model Spearman (mean_pts):     0.377
               feature  beta_pts  beta_IP
                   age     0.493    0.144
                age_sq    -0.477   -0.058
    HLDSV_per_G_recent     0.322   -0.029
              K%_decay     0.180    0.008
            CSW%_decay     0.109    0.063
           RS/9_recent     0.038    0.056
relief_IP_per_G_recent    -0.033   -0.077
             BB%_decay     0.010   -0.013
             FIP_decay    -0.005   -0.068
        delta_IP_per_G    -0.003    0.009
             GB%_decay     0.001    0.024
       IP_per_G_recent     0.001   -0.145


# RP: New Season Predictions

In [29]:
# ── Train on ALL seasons, predict PRED_SEASON ─────────────────────
print(f"Training RP on all data through {PRED_SEASON - 1}...")
rp_train_all = split_by_role(train_all)[1]

X_rp_all, pitcher_idx_rp_all, y_pts_rp_all, y_IP_rp_all, \
    scaler_rp_final, rp_encoder_final, _ = \
    prepare_tensors(rp_train_all, rp_train_all, RP_FEATURES)

n_pitchers_rp_all = len(rp_encoder_final)
rp_guide_final, rp_losses_final = run_inference(
    rp_model, X_rp_all, pitcher_idx_rp_all, n_pitchers_rp_all + 1,
    y_pts_rp_all, y_IP_rp_all)

rp_inference    = build_inference_rows(df, role='RP')
rp_rankings_new = predict_new_season(
    rp_guide_final, rp_model, rp_inference,
    RP_FEATURES, scaler_rp_final, rp_encoder_final, n_pitchers_rp_all, alpha = 0.5)

print(f"\n=== Top 20 RP for {PRED_SEASON} ===")
print(rp_rankings_new.head(20).to_string(index=False))

Training RP on all data through 2025...
  step    0  loss 1730728.6
  step  500  loss 22521.6
  step 1000  loss 21249.8
  step 1500  loss 20748.4
  step 2000  loss 20393.7
  step 2500  loss 19921.3

=== Top 20 RP for 2026 ===
          name  age   mean_pts    std_pts        p10        p90  score  is_known
  Andrés Muñoz   27 120.000000  77.199997  33.200001 219.800003 13.651      True
   Bryan Abreu   29 125.000000  94.900002  20.100000 244.399994 12.824      True
 Fernando Cruz   36 103.099998  85.199997  10.300000 214.600006 11.166      True
 Trevor Megill   33  88.199997  62.700001  14.700000 163.199997 11.135      True
  Tyler Holton   30 115.300003 110.400002 -10.900000 259.100006 10.973      True
  Mason Miller   28 107.599998  96.500000  12.200000 226.199997 10.952      True
   Griffin Jax   32 105.199997  95.000000  12.200000 223.399994 10.792      True
   Jhoan Durán   28  88.900002  74.099998   6.500000 191.199997 10.335      True
Emmanuel Clase   28  92.199997  89.400002   9

In [30]:
sp_rankings_new.head(300).to_csv('./outputs/sp_rankings.csv', index=False)
rp_rankings_new.head(500).to_csv('./outputs/rp_rankings.csv', index=False)